In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "stix",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

CRITICAL = (177/255, 56/255, 69/255)

def _blend(rgb, mix):
    return tuple(c + (1 - c) * mix for c in rgb)

def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

Q       = 0.5
P       = 0.75
N_STEPS = 1000
N_MEAN  = 500
N_THIN  = 20

rng        = np.random.default_rng()
steps_plot = np.arange(0, N_STEPS + 1)
steps_th   = np.arange(2, N_STEPS + 1)

thin_paths = [simulate_erw(N_STEPS, p=P, q=Q, rng=rng).astype(float)**2
              for _ in range(N_THIN)]
mat  = np.array([simulate_erw(N_STEPS, p=P, q=Q, rng=rng).astype(float)**2
                 for _ in range(N_MEAN)])
emp  = mat.mean(axis=0)

asymp = steps_th * np.log(steps_th)

base_col = CRITICAL
darkd    = tuple(c * 0.62 for c in base_col)

fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.0))
fig.subplots_adjust(left=0.14, right=0.97, bottom=0.16, top=0.88)

thin_paths_sorted = sorted(thin_paths, key=lambda s: np.max(s))
maxvals = np.array([np.max(s) for s in thin_paths_sorted])
ranks   = (maxvals - maxvals.min()) / (maxvals.max() - maxvals.min() + 1e-9)

for S2, rank in zip(thin_paths_sorted, ranks):
    mix   = 0.30 - rank * 0.25
    color = _blend(base_col, mix)
    lw    = 0.12 + rank * 0.13
    ax.plot(steps_plot, S2, color=color, lw=lw, alpha=1.0, rasterized=True)

ax.plot(steps_plot, emp,
        color=darkd, lw=0.9, alpha=0.95, zorder=3, solid_capstyle="round")
ax.plot(steps_th, asymp,
        color=darkd, lw=0.9, alpha=0.95, zorder=4,
        ls=(0, (5, 3)), solid_capstyle="round")

ymax = np.nanmax(np.concatenate([emp, asymp, *thin_paths]))
ax.set_xlim(0, N_STEPS)
ax.set_ylim(0, ymax * 1.06)

ax.set_title("$p = 3/4$", fontsize=9.5, pad=5, color="darkgray")
ax.set_xlabel(r"$n$", labelpad=3)
ax.set_ylabel(r"$S_n^2$", labelpad=4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.55)
ax.spines["bottom"].set_linewidth(0.55)
ax.tick_params(width=0.55, length=3, direction="out")

ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${int(x)}$"))
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True))

plt.show()